# Part A

In [1]:
import torch
import torch.nn as nn

torch.manual_seed(42)

names = ["Aridia",
         "Borland",
         "Cyrenia",
         "Deltora",
         "Estovia",
         "Freland",
         "Galenia",
         "Hesperia"
        ]

A_full = torch.tensor([
    #A  B  C  D  E  F  G  H
    [0, 1, 1, 0, 0, 0, 1, 0], # A
    [1, 0, 0, 0, 0, 1, 1, 0], # B
    [1, 0, 0, 0, 0, 1, 1, 0], # C
    [0, 0, 0, 0, 1, 1, 0, 1], # D
    [0, 0, 0, 1, 0, 0, 0, 1], # E
    [0, 1, 1, 1, 0, 0, 1, 1], # F
    [1, 1, 1, 0, 0, 1, 0, 0], # G
    [0, 0, 0, 1, 1, 1, 0, 0] # H
    ], dtype=torch.float32
)

degree = A_full.sum(dim=1)
degree_ave = torch.mean(degree)
unique_edges = torch.sum(degree)/2

X = torch.tensor([
    [5.8, 3.1, 4.2, 46, 6.2],
    [5.1, 3.8, 5.0, 42, 5.7],
    [6.4, 2.7, 3.8, 51, 7.1],
    [3.2, 6.5, 7.4, 30, 3.1],
    [2.8, 7.2, 8.1, 27, 2.8],
    [4.4, 4.1, 5.6, 39, 4.9],
    [5.6, 3.0, 4.5, 48, 6.5],
    [3.5, 5.8, 6.9, 33, 3.6]
    ], dtype=torch.float32
)

X_mean = X.mean(dim=0)
X_std = X.std(dim=0)
X_scaled = (X - X_mean)/X_std

print(f"Diagonal: {torch.diag(A_full).numpy()}")
print(f"Degree of Country: {degree.numpy()}")
print(f"Mean Degree: {degree_ave.numpy()}")
print(f"Unique Edges: {unique_edges.numpy()}")

Diagonal: [0. 0. 0. 0. 0. 0. 0. 0.]
Degree of Country: [3. 3. 3. 3. 2. 5. 4. 3.]
Mean Degree: 3.25
Unique Edges: 13.0


Summing all entries of a symmetric adjacency matrix double-counts undirected edges because the connection is counted on both sides. For example, A--B, B is counted on row A and A is counted on row B which essentially double counts the same connection.

Standardizing the feature matrix is important because each feature can have different scales so larger data in a feature will give that feature a higher weight and standardizing will remove that problem making each feature equally important.

# Part B

In [2]:
# B1
A_train = torch.tensor([
    #A  B  C  D  E  F  G  H
    [0, 1, 1, 0, 0, 0, 0, 0], # A
    [1, 0, 0, 0, 0, 1, 1, 0], # B
    [1, 0, 0, 0, 0, 1, 1, 0], # C
    [0, 0, 0, 0, 1, 1, 0, 0], # D
    [0, 0, 0, 1, 0, 0, 0, 1], # E
    [0, 1, 1, 1, 0, 0, 1, 1], # F
    [0, 1, 1, 0, 0, 1, 0, 0], # G
    [0, 0, 0, 0, 1, 1, 0, 0] # H
    ], dtype=torch.float32
)

train_degree = A_train.sum(dim=1)
train_unique_edges = torch.sum(train_degree)/2
print(f"Unique Edges: {train_unique_edges.numpy()}")
print(f"Training graph contains 11 unique edges: {train_unique_edges.numpy() == 11.0}")


Unique Edges: 11.0
Training graph contains 11 unique edges: True


In [3]:
# B2
def aggregate(A, H):
    I = torch.eye(A.shape[0])
    A_hat = A + I
    degree_hat = A_hat.sum(dim=1)
    degree_hat = degree_hat.clamp(min=1) # handle degrees safely, basically does max(degree_hat, 1)
    D_inverse = torch.diag(1.0 / degree_hat)
    X_aggregated = D_inverse @ A_hat @ H
    assert X_aggregated.shape == H.shape, "X_aggregated's shape == H.shape"
    # print(f"X_aggregated has the same shape as H: {X_aggregated.shape == H.shape}")
    return X_aggregated

In [4]:
# B3
class GNNEncoder(nn.Module):
    def __init__(self, input_features=5, hidden_size=8, embedding_size=3):
        super().__init__()
        self.linear_one = nn.Linear(input_features, hidden_size)
        self.activation = nn.ReLU()
        self.linear_two = nn.Linear(hidden_size, embedding_size)

    def forward(self, A, X):
        layer_one = self.linear_one(aggregate(A, X))
        X_activated = self.activation(layer_one)
        layer_two = self.linear_two(aggregate(A, X_activated))
        return layer_two

class GraphAutoencoder(nn.Module):
    def __init__(self, input_features_size, hidden_size, embedding_size):
        super().__init__()
        self.encoder = GNNEncoder(input_features_size, hidden_size, embedding_size)

    def forward(self, A, X):
        Z = self.encoder(A, X)
        scores = Z @ Z.T
        A_reconstructed = torch.sigmoid(scores)
        return A_reconstructed, Z


def report(model, A, X):
    model.eval()
    with torch.no_grad():
        probabilities, embeddings = model(A, X)
        # rows, cols = unique_pairs(A)
        # loss = nn.functional.binary_cross_entropy(
        #     probabilities[rows, cols], A[rows, cols]
        # ).item()
        # accuracy = (
        #     (probabilities[rows, cols] >= 0.5) == A[rows, cols].bool()
        # ).float().mean().item()

    # print("Supplied adjacency matrix:")
    # print(A)
    print("Reconstructed edge probabilities (diagonal NOT forced to zero):")
    print(probabilities)

    # print("Reconstructed adjacency at threshold 0.5:")
    # print((probabilities >= 0.5).int())
    # print(f"Pairwise BCE: {loss:.4f}; pairwise accuracy: {accuracy:.0%}")
    # print("Edge                 Target    Score")
    # for i, j in zip(rows.tolist(), cols.tolist()):
    #     print(f"{names[i] + '--' + names[j]:20s}"
    #         f" {int(A[i, j]):3d}      {probabilities[i, j]:.4f}")

    print("\nCountry: Embedding")
    for i, embedding in enumerate(embeddings):
        print(f"{names[i]}: {embedding}")

    return probabilities, embeddings



In [5]:
# B4
torch.manual_seed(42)
INPUT_FEATURES = X_scaled
ENCODER_GRAPH = A_train
BASELINE_RECONSTRUCTION_TARGET = A_train
HIDDEN_DIMENSION = 8
EMBEDDING_DIMENSION = 3
LEARNING_RATE = 0.01
EPOCHS = 2000

model = GraphAutoencoder(INPUT_FEATURES.shape[1], HIDDEN_DIMENSION, EMBEDDING_DIMENSION)
loss_function = nn.BCELoss()
optimizer = torch.optim.Adam(
    model.parameters(), lr=LEARNING_RATE
)

for epoch in range(EPOCHS):
    optimizer.zero_grad()
    A_reconstructed, Z = model(ENCODER_GRAPH, INPUT_FEATURES)
    loss = loss_function(A_reconstructed, BASELINE_RECONSTRUCTION_TARGET)
    loss.backward()
    optimizer.step()

    if epoch % 200 == 0:
        print(f"Epoch {epoch}: loss = {loss}")

    if epoch == EPOCHS - 1:
        print(f"Final Loss: {loss}\n")

report(model, ENCODER_GRAPH, INPUT_FEATURES)

Epoch 0: loss = 0.7324681282043457
Epoch 200: loss = 0.5990166664123535
Epoch 400: loss = 0.5987579226493835
Epoch 600: loss = 0.5982092618942261
Epoch 800: loss = 0.5979307293891907
Epoch 1000: loss = 0.5976468324661255
Epoch 1200: loss = 0.5973796248435974
Epoch 1400: loss = 0.597136914730072
Epoch 1600: loss = 0.5967106223106384
Epoch 1800: loss = 0.596149742603302
Final Loss: 0.594852864742279

Reconstructed edge probabilities (diagonal NOT forced to zero):
tensor([[0.7074, 0.7010, 0.7080, 0.2971, 0.2053, 0.5670, 0.7147, 0.3202],
        [0.7010, 0.7166, 0.7159, 0.2930, 0.1920, 0.5890, 0.7399, 0.3239],
        [0.7080, 0.7159, 0.7179, 0.2897, 0.1909, 0.5833, 0.7362, 0.3184],
        [0.2971, 0.2930, 0.2897, 0.7034, 0.7993, 0.4231, 0.2746, 0.6767],
        [0.2053, 0.1920, 0.1909, 0.7993, 0.9036, 0.3689, 0.1671, 0.7624],
        [0.5670, 0.5890, 0.5833, 0.4231, 0.3689, 0.5440, 0.6061, 0.4405],
        [0.7147, 0.7399, 0.7362, 0.2746, 0.1671, 0.6061, 0.7678, 0.3105],
        [0.3202,

(tensor([[0.7074, 0.7010, 0.7080, 0.2971, 0.2053, 0.5670, 0.7147, 0.3202],
         [0.7010, 0.7166, 0.7159, 0.2930, 0.1920, 0.5890, 0.7399, 0.3239],
         [0.7080, 0.7159, 0.7179, 0.2897, 0.1909, 0.5833, 0.7362, 0.3184],
         [0.2971, 0.2930, 0.2897, 0.7034, 0.7993, 0.4231, 0.2746, 0.6767],
         [0.2053, 0.1920, 0.1909, 0.7993, 0.9036, 0.3689, 0.1671, 0.7624],
         [0.5670, 0.5890, 0.5833, 0.4231, 0.3689, 0.5440, 0.6061, 0.4405],
         [0.7147, 0.7399, 0.7362, 0.2746, 0.1671, 0.6061, 0.7678, 0.3105],
         [0.3202, 0.3239, 0.3184, 0.6767, 0.7624, 0.4405, 0.3105, 0.6553]]),
 tensor([[ 0.8423, -0.3323, -0.2511],
         [ 0.9536, -0.1344, -0.0168],
         [ 0.9378, -0.2102, -0.1022],
         [-0.8882,  0.2361,  0.1374],
         [-1.4667,  0.2737,  0.1081],
         [ 0.3897,  0.0706,  0.1395],
         [ 1.0885, -0.0668,  0.0817],
         [-0.7310,  0.2661,  0.1933]]))

# Part C

In [6]:
# C1
aridia, borland, cyrenia, deltora, estovia, freland, galenia, hesperia = range(8)

with torch.no_grad():
    _, Z = model(ENCODER_GRAPH, INPUT_FEATURES)

    aridia_galenia = torch.sigmoid(Z[aridia] @ Z[galenia])
    aridia_estovia = torch.sigmoid(Z[aridia] @ Z[estovia])
    deltora_hesperia = torch.sigmoid(Z[deltora] @ Z[hesperia])
    deltora_cyrenia = torch.sigmoid(Z[deltora] @ Z[cyrenia])

    print(f"Aridia - Galenia: {aridia_galenia}")
    print(f"Aridia - Estovia: {aridia_estovia}")
    print(f"Hidden real link outranks non-edge: {aridia_galenia > aridia_estovia}\n")

    print(f"Deltora - Hesperia: {deltora_hesperia}")
    print(f"Deltora - Cyrenia: {deltora_cyrenia}")
    print(f"Hidden real link outranks non-edge: {deltora_hesperia > deltora_cyrenia}")


Aridia - Galenia: 0.714745283126831
Aridia - Estovia: 0.20529133081436157
Hidden real link outranks non-edge: True

Deltora - Hesperia: 0.6766974925994873
Deltora - Cyrenia: 0.2897382974624634
Hidden real link outranks non-edge: True


In [7]:
# C2

def unique_pairs(A):
    # Undirected edges occur twice in A. Score each pair once, excluding the
    # diagonal: self-loops support aggregation but are not prediction targets.
    return torch.triu_indices(
        A.shape[0], A.shape[1], offset=1,
        device=A.device,
    )

rows, cols = unique_pairs(A_train)
reconstructed_scores = []
with torch.no_grad():
    probabilities, Z = model(ENCODER_GRAPH, INPUT_FEATURES)

    # Because pairs were appended in index order and because Python's sorting is stable,
    # pairs with tied scores stay in increasing pair index order.
    for i, j in zip(rows.tolist(), cols.tolist()):
        if A_train[i, j] == 0:
            reconstructed_scores.append((probabilities[i, j], i, j))
    reconstructed_scores = sorted(reconstructed_scores, key=lambda x:x[0], reverse=True)

for idx, score in enumerate(reconstructed_scores):
    if score[1] == aridia and score[2] == galenia:
        print(f"Aridia - Galena is Rank {idx + 1}")
    if score[1] == deltora and score[2] == hesperia:
        print(f"Deltora - Hesperia is Rank {idx + 1}\n")

for i in range(5):
    print(f"{names[reconstructed_scores[i][1]]} - {names[reconstructed_scores[i][2]]}: {reconstructed_scores[i][0]}")


Aridia - Galena is Rank 2
Deltora - Hesperia is Rank 3

Borland - Cyrenia: 0.7159192562103271
Aridia - Galenia: 0.714745283126831
Deltora - Hesperia: 0.6766974925994873
Aridia - Freland: 0.5670201182365417
Estovia - Freland: 0.36885765194892883


In [8]:
# C3
country_to_closest_country = {}

with torch.no_grad():
    _, Z = model(ENCODER_GRAPH, INPUT_FEATURES)
    # https://docs.pytorch.org/docs/main/generated/torch.cdist.html
    distances = torch.cdist(Z, Z)

    for i in range(len(distances)):
        closest = float('inf')
        for j in range(len(distances[i])):
            if i == j: continue
            if distances[i, j] < closest:
                closest = distances[i, j]
                country_to_closest_country[i] = (j, closest)

for country in country_to_closest_country:
    print(f"The closest country to {names[country]} is {names[country_to_closest_country[country][0]]}. Distance: {country_to_closest_country[country][1]:4f}")


with torch.no_grad():
    _, Z = model(ENCODER_GRAPH, INPUT_FEATURES)

    centroid = Z.mean(dim=0)

    centroid_distances = torch.norm(Z - centroid, dim=1)

# rank descending (largest distance = most "anomalous");
# stable sort so ties resolve in country-index order
anomaly_ranking = sorted(
    range(len(names)), key=lambda i: centroid_distances[i].item(), reverse=True
)

print(f"\nEmbedding centroid: {centroid}\n")
print("Rank  Country     Distance to centroid")
for rank, i in enumerate(anomaly_ranking, start=1):
    print(f"{rank:<5} {names[i]:<11} {centroid_distances[i]:.4f}")





The closest country to Aridia is Cyrenia. Distance: 0.214981
The closest country to Borland is Cyrenia. Distance: 0.115258
The closest country to Cyrenia is Borland. Distance: 0.115258
The closest country to Deltora is Hesperia. Distance: 0.169523
The closest country to Estovia is Deltora. Distance: 0.580499
The closest country to Freland is Borland. Distance: 0.620102
The closest country to Galenia is Borland. Distance: 0.180177
The closest country to Hesperia is Deltora. Distance: 0.169523

Embedding centroid: tensor([0.1408, 0.0129, 0.0362])

Rank  Country     Distance to centroid
1     Estovia     1.6301
2     Deltora     1.0577
3     Galenia     0.9522
4     Hesperia    0.9213
5     Cyrenia     0.8392
6     Aridia      0.8330
7     Borland     0.8278
8     Freland     0.2756


The centroid distance shows the countries the model represents differently from the rest. It is a pattern in the embedding space, shaped by the objective of the training, the sizes of the clusters, and the overall structure of the network. This heuristic alone does not contain real economic criteria to call a country "economically abnormal."

### Part D

In [9]:
# D1

X_changed = X.clone()
X_changed[borland, 1] = 12.0

X_changed_scaled = (X_changed - X_mean)/X_std

with torch.no_grad():
    Z_before = model.encoder(A_train, X_scaled)
    Z_after  = model.encoder(A_train, X_changed_scaled)

    print("Country: ||Z_after - Z_before||_2")
    for c in [borland, aridia, freland, galenia, estovia]:
        change = torch.norm(Z_after[c] - Z_before[c])
        print(f"{names[c]:>7}: {change:.4f}")


Country: ||Z_after - Z_before||_2
Borland: 4.3744
 Aridia: 2.8754
Freland: 2.2030
Galenia: 3.3044
Estovia: 0.0000


Only Borland's inflation was changed but the embeddings of other countries also moved. Each GNN layer averages a country's features with its trading partners so the change in Borland spreads through the links in A_train. Borland changed the most (4.37) since the change is in its own features. Its direct neighbors Galenia (3.29), Aridia (2.87), and Freland (2.20) changed as well, but Freland changed the least out of the three because it has 5 trading partners so Borland's features get averaged out more. Estovia did not change at all (0.0) because it is 3 hops away from Borland (Borland -> Freland -> Deltora -> Estovia) and a 2 layer GNN can only reach 2 hops. This is also why a non-neighbor can still change if it is 2 hops away, for example Cyrenia through Aridia.

In [10]:
# D2
class BaselineEncoder(nn.Module):
    def __init__(self, input_features=5, hidden_size=8, embedding_size=3):
        super().__init__()
        self.linear_one = nn.Linear(input_features, hidden_size)
        self.activation = nn.ReLU()
        self.linear_two = nn.Linear(hidden_size, embedding_size)

    def forward(self, X):
        return self.linear_two(self.activation(self.linear_one(X)))

torch.manual_seed(42)
baseline = BaselineEncoder(INPUT_FEATURES.shape[1], HIDDEN_DIMENSION, EMBEDDING_DIMENSION)

with torch.no_grad():
    Z_baseline = baseline(X_scaled)
    print(f"Baseline embedding shape: {Z_baseline.shape}")



Baseline embedding shape: torch.Size([8, 3])


The BaselineEncoder only receives X_scaled so each country is represented by its own 5 features only. The GNN, on the other hand, also has the adjacency matrix A_train, which lets it average each country's features with its trading partners, and with 2 layers it can reach countries up to 2 hops away. The baseline does not know who trades with who. For example, if the Borland shock from D1 was passed into the baseline, only Borland's embedding would change and Aridia, Galenia, and Freland would stay the same since nothing connects them to Borland. The baseline here is also untrained so its weights are still random and its embeddings do not mean anything yet. To compare it with the GNN properly, both would need to be trained on the same objective (sigmoid(ZZ^T) with BCE on A_train) and then compared using things like the final loss and the ranks of the withheld links.

### Part E

In [17]:
A_train = torch.tensor([
    #A  B  C  D  E  F  G  H
    [0, 1, 1, 0, 0, 0, 0, 0], # A
    [1, 0, 0, 0, 0, 1, 1, 0], # B
    [1, 0, 0, 0, 0, 1, 1, 0], # C
    [0, 0, 0, 0, 1, 1, 0, 0], # D
    [0, 0, 0, 1, 0, 0, 0, 1], # E
    [0, 1, 1, 1, 0, 0, 1, 1], # F
    [0, 1, 1, 0, 0, 1, 0, 0], # G
    [0, 0, 0, 0, 1, 1, 0, 0] # H
    ], dtype=torch.float32
)

def aggregate_symmetric(A, H):
    I = torch.eye(A.shape[0])
    A_hat = A + I
    degree_hat = A_hat.sum(dim=1)
    degree_hat = degree_hat.clamp(min=1) # handle degrees safely, basically does max(degree_hat, 1)
    d_inv_sqrt = torch.pow(degree_hat, -0.5)
    d_inv_sqrt[torch.isinf(d_inv_sqrt)] = 0.0
    D_inv_sqrt = torch.diag(d_inv_sqrt)
    X_aggregated = D_inv_sqrt @ A @ D_inv_sqrt @ H
    assert X_aggregated.shape == H.shape, "X_aggregated's shape == H.shape"
    # print(f"X_aggregated has the same shape as H: {X_aggregated.shape == H.shape}")

    return X_aggregated

class GNNEncoder(nn.Module):
    def __init__(self, input_features=5, hidden_size=8, embedding_size=3):
        super().__init__()
        self.linear_one = nn.Linear(input_features, hidden_size)
        self.activation = nn.ReLU()
        self.linear_two = nn.Linear(hidden_size, embedding_size)

    def forward(self, A, X):
        layer_one = self.linear_one(aggregate_symmetric(A, X))
        X_activated = self.activation(layer_one)
        layer_two = self.linear_two(aggregate_symmetric(A, X_activated))
        return layer_two

class GraphAutoencoder(nn.Module):
    def __init__(self, input_features_size, hidden_size, embedding_size):
        super().__init__()
        self.encoder = GNNEncoder(input_features_size, hidden_size, embedding_size)

    def forward(self, A, X):
        Z = self.encoder(A, X)
        scores = Z @ Z.T
        A_reconstructed = torch.sigmoid(scores)
        return A_reconstructed, Z

torch.manual_seed(42)
INPUT_FEATURES = X_scaled
ENCODER_GRAPH = A_train
BASELINE_RECONSTRUCTION_TARGET = A_train
HIDDEN_DIMENSION = 8
EMBEDDING_DIMENSION = 3
LEARNING_RATE = 0.01
EPOCHS = 2000

model_new = GraphAutoencoder(INPUT_FEATURES.shape[1], HIDDEN_DIMENSION, EMBEDDING_DIMENSION)
loss_function = nn.BCELoss()
optimizer = torch.optim.Adam(
    model_new.parameters(), lr=LEARNING_RATE
)

for epoch in range(EPOCHS):
    optimizer.zero_grad()
    A_reconstructed, Z = model_new(ENCODER_GRAPH, INPUT_FEATURES)
    loss = loss_function(A_reconstructed, BASELINE_RECONSTRUCTION_TARGET)
    loss.backward()
    optimizer.step()

    if epoch % 200 == 0:
        print(f"Epoch {epoch}: loss = {loss}")

    if epoch == EPOCHS - 1:
        print(f"Final Loss: {loss}\n")

report(model_new, ENCODER_GRAPH, INPUT_FEATURES)

Epoch 0: loss = 0.7428263425827026
Epoch 200: loss = 0.596309244632721
Epoch 400: loss = 0.5960593819618225
Epoch 600: loss = 0.5958184003829956
Epoch 800: loss = 0.5953010320663452
Epoch 1000: loss = 0.5919642448425293
Epoch 1200: loss = 0.5819215178489685
Epoch 1400: loss = 0.5798442959785461
Epoch 1600: loss = 0.5798985958099365
Epoch 1800: loss = 0.5798218250274658
Final Loss: 0.5798214673995972

Reconstructed edge probabilities (diagonal NOT forced to zero):
tensor([[0.8486, 0.6859, 0.6859, 0.2489, 0.2486, 0.3134, 0.5469, 0.2489],
        [0.6859, 0.7352, 0.7352, 0.2940, 0.1870, 0.6209, 0.7599, 0.2940],
        [0.6859, 0.7352, 0.7352, 0.2940, 0.1870, 0.6209, 0.7599, 0.2940],
        [0.2489, 0.2940, 0.2940, 0.7149, 0.7779, 0.5063, 0.3272, 0.7149],
        [0.2486, 0.1870, 0.1870, 0.7779, 0.8923, 0.3256, 0.1586, 0.7779],
        [0.3134, 0.6209, 0.6209, 0.5063, 0.3256, 0.8079, 0.7810, 0.5063],
        [0.5469, 0.7599, 0.7599, 0.3272, 0.1586, 0.7810, 0.8489, 0.3272],
        [0.248

(tensor([[0.8486, 0.6859, 0.6859, 0.2489, 0.2486, 0.3134, 0.5469, 0.2489],
         [0.6859, 0.7352, 0.7352, 0.2940, 0.1870, 0.6209, 0.7599, 0.2940],
         [0.6859, 0.7352, 0.7352, 0.2940, 0.1870, 0.6209, 0.7599, 0.2940],
         [0.2489, 0.2940, 0.2940, 0.7149, 0.7779, 0.5063, 0.3272, 0.7149],
         [0.2486, 0.1870, 0.1870, 0.7779, 0.8923, 0.3256, 0.1586, 0.7779],
         [0.3134, 0.6209, 0.6209, 0.5063, 0.3256, 0.8079, 0.7810, 0.5063],
         [0.5469, 0.7599, 0.7599, 0.3272, 0.1586, 0.7810, 0.8489, 0.3272],
         [0.2489, 0.2940, 0.2940, 0.7149, 0.7779, 0.5063, 0.3272, 0.7149]]),
 tensor([[ 1.2594, -0.2705,  0.2534],
         [ 0.5691,  0.4320,  0.7147],
         [ 0.5691,  0.4320,  0.7147],
         [-0.8064, -0.1389, -0.4997],
         [-0.8055, -0.6300, -1.0339],
         [-0.5757,  0.8273,  0.6487],
         [ 0.1348,  0.8567,  0.9868],
         [-0.8064, -0.1389, -0.4997]]))

We can observe that the model that used symmetric normalization (0.57) has a lower final loss compared to the model that used row-mean aggregation (0.59) so it is better at reconstructing the given training data. Comparing the predictions of the two models on the link between the withheld links, the original model showed A:G(0.7147) and D:H(0.6767) and the new model showed A:G(0.5469) and D:H(0.7149), we can see that the original model is more confident in A and G linkage but the new model is more confident in the link between D and H but less confident in the other. A lower reconstruction loss does not necessarily mean better hidden-link rankings because these models are trained to reconstruct the original dataset which can harm the ratings between the withheld links.